# Разведочный анализ данных (Exploratory Data Analysis, EDA) в Python

В этом ноутбуке на практическом примере наблюдений метеостанции Малые Кармакулы разбираются базовые приёмы разведочного анализа данных (EDA) в Python.

Основные инструменты — библиотеки **pandas** (табличные данные и статистика), **NumPy** (числовые массивы), **Matplotlib** и **Seaborn** (визуализация) и **Plotly** (интерактивные графики); ниже для каждого раздела указаны конкретные функции со ссылками на документацию.

## Зачем нужен разведочный анализ как первый этап решения задачи машинного обучения
- Понимание структуры, распределений и особенностей данных
- Выявление аномалий, ошибок и пропусков
- Выявление взаимосвязей между признаками
- Выбор признаков
- Понимание необходимости порождения новых признаков на основе существующих (feature engineering)



## Полезные ссылки
- <a href="https://pandas.pydata.org/">Документация Pandas</a>
- <a href="https://matplotlib.org/">Документация Matplotlib</a>
- <a href="https://seaborn.pydata.org/">Документация Seaborn</a>
- <a href="https://numpy.org/doc/stable/">Документация NumPy</a>
- <a href="https://plotly.com/python/">Документация Plotly</a>

## Дополнительные материалы

- Маккинни У. Python и анализ данных / Пер. с англ. Слинкин А.А. – М: ДМК Пресс, 2015. – 482 с. ([Google Books]
- Брюс П., Брюс Э. Практическая статистика для специалистов Data Science. – БХВ-Петербург, 2018. — 304 с. ([Google Books](https://books.google.ru/books?hl=ru&lr=&id=l_6MDwAAQBAJ&oi=fnd&pg=PA5&dq=%D0%BF%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F+%D1%81%D1%82%D0%B0%D1%82%D0%B8%D1%81%D1%82%D0%B8%D0%BA%D0%B0+%D0%B4%D0%BB%D1%8F+%D1%81%D0%BF%D0%B5%D1%86%D0%B8%D0%B0%D0%BB%D0%B8%D1%81%D1%82%D0%BE%D0%B2+data+science&ots=fB2sdc0NnS&sig=S7_kC8Nv2Ipg5By2UbTDVDGVvqE&redir_esc=y#v=onepage&q=%D0%BF%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F%20%D1%81%D1%82%D0%B0%D1%82%D0%B8%D1%81%D1%82%D0%B8%D0%BA%D0%B0%20%D0%B4%D0%BB%D1%8F%20%D1%81%D0%BF%D0%B5%D1%86%D0%B8%D0%B0%D0%BB%D0%B8%D1%81%D1%82%D0%BE%D0%B2%20data%20science&f=false))
- [Лекции с YouTube](https://www.youtube.com/watch?v=Yhrp-YVZH84)
- [Еще лекции с YouTube](https://youtu.be/uJpDzHGUamg?si=29xzZdo2AzE12nOH)


In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns


## Загрузка данных

Будем работать с данными метеостанции Малые Кармакулы из [архива ВНИИГМИ-МЦД](http://meteo.ru/data/): сначала читаем файл с описанием полей, затем — сам файл наблюдений.

**Инструменты:** [pandas.read_fwf](https://pandas.pydata.org/docs/reference/api/pandas.read_fwf.html) — чтение файла с колонками фиксированной ширины, [pandas.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) — чтение таблицы с разделителем «;».

In [ ]:
data_path = 'EDA_demo_data/wr282025.txt'
fields_path = 'EDA_demo_data/fld282025a0.txt'

# Загружаем информацию о полях из файла данных ВНИИГМИ-МЦД c фиксированной шириной колонок
df_fields = pd.read_fwf (fields_path, encoding = "cp1251", header = None)

display (df_fields)


In [ ]:

df_fields[3] = [' '.join(field.split()) for field in df_fields[3]]
display (df_fields[3])


In [ ]:
df = pd.read_csv (data_path, sep=';', header=None, names=df_fields[3])
df.head()


### Подготовка данных

Приводим данные к удобному виду: собираем единую метку времени, переименовываем колонки и делаем её индексом.

**Инструменты:** [pandas.Timestamp](https://pandas.pydata.org/docs/reference/api/pandas.Timestamp.html), [DataFrame.drop](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html), [DataFrame.set_index](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.set_index.html), [DataFrame.rename](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html).

In [ ]:
df['datetime'] = [pd.Timestamp(year,month,day,hour) for year,month,day,hour in zip(df['Год по Гринвичу'], df['Месяц по Гринвичу'], df['День по Гринвичу'], df['Срок по Гринвичу'])]

df = df.drop (columns=[x for x in df.columns if 'по Гринвичу' in x or 'Синоптический индекс' in x])

df = df.set_index ('datetime')

new_columns = {'Общее количество облачности': 'tcc', 
               'Количество облачности нижнего яруса': 'lcc', 
               'Направление ветра': 'wdir', 
               'Средняя скорость ветра': 'wvel', 
               'Максимальная скорость ветра': 'wgust',
               'Сумма осадков': 'prec', 
               'Температура поверхности почвы': 'tg', 
               'Температура воздуха по сухому терм-ру': 'ta', 
               'Относительная влажность воздуха': 'rh', 
               'Температура точки росы': 'td', 
               'Атмосферное давление на уровне станции': 'ps', 
               'Атмосферное давление на уровне моря': 'psl'}

df = df.rename(columns=new_columns)
display(df.head())


### Анализ типов данных

Проверяем типы данных и пропуски, а затем приводим все значения к числовому типу и сохраняем очищенную таблицу в CSV.

**Инструменты:** [DataFrame.info](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html), [pandas.to_numeric](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html), [DataFrame.apply](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html), [DataFrame.to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html).

In [ ]:
display(df.info())



In [ ]:
# Преобразуем текстовые значения в числовые
df = df.apply(pd.to_numeric, errors='coerce')
display(df.info())

# Сохраним результаты в новый csv-файл
df.to_csv('EDA_demo_data/karmakuly.csv')


## Визуализация временных рядов

Строим графики скорости ветра для эпизода боры 10–12 декабря 2006 г. из статьи [(Shestakova and Debolskiy, 2022)](https://www.mdpi.com/2073-4433/13/7/1108).

**Инструменты:** [DataFrame.plot](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html) (бэкенд Matplotlib) и функции [matplotlib.pyplot](https://matplotlib.org/stable/api/pyplot_summary.html) (`figure`, `xlim`, `gca`).

In [ ]:
%matplotlib inline 
#%matplotlib qt
plt.rcParams['figure.dpi'] = 100

plt.figure()
df['09-Dec-2006':'13-Dec-2006'][['wvel', 'wgust']].plot(ax = plt.gca(), xlabel = 'Дата', ylabel = 'Скорость ветра, м/с')
plt.title ('Бора 09-13.12.2006')

plt.figure()
df[['wvel', 'wgust']].plot(ax = plt.gca(), xlabel = 'Дата', ylabel = 'Скорость ветра, м/с')
plt.xlim(('09-Dec-2006','13-Dec-2006'))


### Интерактивный график с помощью Plotly

Строим интерактивный график скорости ветра за тот же эпизод с помощью Plotly.

**Инструменты:** переключение бэкенда [pandas.options.plotting.backend](https://pandas.pydata.org/docs/user_guide/visualization.html#plotting-backends), сами интерактивные фигуры [Plotly](https://plotly.com/python/), настройка осей через `update_layout` и вывод через `fig.show()` — см. [создание и обновление фигур Plotly](https://plotly.com/python/creating-and-updating-figures/).

In [ ]:
pd.options.plotting.backend = "plotly"

fig = df['2006':'2007'][['wvel', 'wgust']].plot()

fig.update_layout(xaxis_range=['2006-12-09', '2006-12-13'], title = 'Скорость ветра (Plotly)',
                  xaxis_title = 'Дата', yaxis_title = 'Скорость ветра, м/с')
fig.show()

pd.options.plotting.backend = "matplotlib"


## Доступ к строкам и столбцам в DataFrame

Показываем разные способы выборки строк и столбцов из DataFrame.

**Инструменты:** [DataFrame.loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) — выборка по меткам, [DataFrame.iloc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iloc.html) — выборка по позициям, а также булева индексация (см. [руководство по индексации](https://pandas.pydata.org/docs/user_guide/indexing.html)).

In [ ]:
# Доступ по индексу
df['2010-01-01 03']


In [ ]:
# Доступ по порядковому номеру
df[10000::8]

In [ ]:
# Доступ по массиву булевских значений
df[df['wgust'] > 30]

In [ ]:
# Доступ к колонкам по имени
df[['wdir', 'wvel']].head()

In [ ]:
# Одновременная обрезка по колонкам и столбцам

#df.loc['2010-01-01 03']
#df.loc['2010-01-01 03':'2010-01-01 12', ['wdir', 'wvel']]
df.iloc[3333, 4:7]

## Анализ доступности данных

Строим карту пропусков по месяцам, чтобы оценить полноту и однородность наблюдений.

**Инструменты:** [DataFrame.isnull](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isnull.html), [DataFrame.resample](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html) и [matplotlib.pyplot.pcolormesh](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.pcolormesh.html).

In [ ]:
%matplotlib inline

data_missing = df.isnull ().astype(float).resample('M', label='left').mean()

#display(data_missing)

plt.figure()
plt.pcolormesh(data_missing.index, data_missing.columns, data_missing.T.to_numpy())
plt.xlabel ('Дата')
plt.ylabel ('Признак')
plt.title ('Доля пропусков по месяцам')
plt.colorbar (label = 'Доля пропусков')
_ = plt.xticks(rotation=45)

#sys.exit()


### Визуализация пропусков с помощью missingno

Дополнительно смотрим структуру и взаимосвязь пропусков с помощью библиотеки missingno.

**Инструменты:** [missingno](https://github.com/ResidentMario/missingno) — функции `matrix`, `bar`, `heatmap` (см. также [пример использования](https://www.geeksforgeeks.org/machine-learning/python-visualize-missing-values-nan-values-using-missingno-library/)).

In [ ]:
import missingno as msno

plt.rcParams['figure.dpi'] = 150

# Матрица пропусков: белые линии соответствуют пропущенным значениям
msno.matrix (df)
plt.show()

# Столбчатая диаграмма полноты данных по столбцам
msno.bar (df)
plt.show()

# Тепловая карта корреляции пропусков между признаками
msno.heatmap (df)
plt.show()

## Одномерный анализ данных (univariate analysis)

Изучаем каждый признак по отдельности: его статистики, распределение и характерные значения.

**Инструменты:** методы и функции [pandas](https://pandas.pydata.org/docs/reference/index.html), [Matplotlib](https://matplotlib.org/stable/api/index.html) и [Seaborn](https://seaborn.pydata.org/api.html); конкретные вызовы перечислены в подразделах ниже.

### Анализ статистических моментов

Смотрим основные описательные статистики (среднее, медиану, квартили, минимум и максимум) для всех признаков.

**Инструменты:** [DataFrame.describe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html) и свойство [DataFrame.T](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.transpose.html) для транспонирования таблицы.

In [ ]:
#df['wvel'].median()
#df.describe()
df.describe().T #Транспонирование

### Анализ уникальных значений

Проверяем, какие значения и сколько раз встречаются в каждом столбце.

**Инструменты:** [pandas.unique](https://pandas.pydata.org/docs/reference/api/pandas.unique.html), [DataFrame.nunique](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nunique.html), [Series.value_counts](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html), а также [DataFrame.plot.bar](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.bar.html) и [DataFrame.plot.pie](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.pie.html).

In [ ]:
nunique = lambda x: len (pd.unique(x))

print ('Уникальные значения в столбцах: ')
display(df.apply (pd.unique))

print ('Число уникальных значений в столбцах:  ')
display(df.nunique())

In [ ]:
# Уникальные значения балла облачности
np.sort (df['tcc'].unique())

In [ ]:
# Подсчет уникальных значений

plt.rcParams['figure.dpi'] = 100

#df_cr = df['2015':]

display(df['tcc'].value_counts().sort_index())
plt.figure()
ax = df['tcc'].value_counts().sort_index().plot.bar(xlabel = 'Балл облачности', ylabel = 'Число случаев') #
ax.set_xticklabels ([f'{float (t.get_text ()):g}' for t in ax.get_xticklabels ()], rotation = 45)
plt.figure()
df['tcc'].value_counts().sort_index().plot.pie() #
plt.title ('Повторяемость балла облачности')

### Анализ функций распределения

Строим гистограммы распределений всех признаков с нормировкой на плотность вероятности.

**Инструменты:** [DataFrame.hist](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.hist.html); аналогичные графики в Seaborn строит [seaborn.histplot](https://seaborn.pydata.org/generated/seaborn.histplot.html).

In [ ]:
#Гистограммы распределения
%matplotlib inline
df.hist(figsize=(20,20), density=True)
plt.suptitle ('Распределения признаков (плотность вероятности)')
[ax.set_ylabel ('Плотность') for ax in plt.gcf().axes]

### Анализ направления ветра

Ищем аномальные и кодовые значения направления ветра (например, 999) вместе с соседними наблюдениями.

**Инструменты:** булева индексация и [Series.shift](https://pandas.pydata.org/docs/reference/api/pandas.Series.shift.html) для получения соседних строк.

In [ ]:
df[df['wdir'] == 999]

In [ ]:
mask_999 = df['wdir'] == 999

# Создаем маски для соседних строк
mask_prev = mask_999.shift(1, fill_value=False)  # предыдущие строки
mask_next = mask_999.shift(-1, fill_value=False)  # следующие строки

# Объединяем все маски
combined_mask = mask_999 | mask_prev | mask_next

result = df[combined_mask]
display(result)

### Графики ядерной плотности (kernel density plots)

Оцениваем плотность распределения скорости ветра с помощью ядерных оценок (KDE) и сравниваем её с гистограммами.

Ядерные оценки плотности — KDE, Kernel Density Estimates — способ что-то понять о распределении, когда неизвестно ничего. Такого рода методы называют непараметрическими, они иллюстрируют разницу подходов в статистике и теории вероятностей: если в теории вероятности известно распределение и исследуются его свойства, то в статистике зачастую известны только данные, и по их свойствам угадывается распределение.

Смысл: в каждую точку выборки поставили отмасштабированное ядро так, будто эта точка — центр ядра, а затем усреднили значения соседних точек с весами, заданными этим ядром. Вместо тысячи слов — [интерактивная иллюстрация](https://mathisonian.github.io/kde/).

**Инструменты:** [DataFrame.plot.hist](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.hist.html) и [DataFrame.plot.density](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.density.html); в Seaborn — [seaborn.kdeplot](https://seaborn.pydata.org/generated/seaborn.kdeplot.html).

In [ ]:
#Гистограммы и графики ядерной плотности 

plt.figure()
df[['wvel', 'wgust']].plot.hist(alpha=0.5, density = True, bins=np.arange(-0.5, 40.5),
                                xlabel = 'Скорость ветра, м/с', ylabel = 'Плотность вероятности')
df[['wvel', 'wgust']].plot.density(ax = plt.gca(), color = ['blue', 'red'], bw_method=0.5) 
plt.xlim(0, 40)


In [ ]:
decades = np.arange(1970, 2010, 10)

for decade_start in decades:
    df4decade = df[str(decade_start) + '-01-01':str(decade_start + 10) + '-12-31']

    plt.figure(figsize=(8,4))
    plt.subplot(1,2,1)
    ax = df4decade['wvel'].value_counts(normalize = True).sort_index().plot.bar(ax = plt.gca(), color = 'blue', alpha = 0.5, label = 'wvel')
    ax.set_xticklabels ([f'{float (t.get_text ()):g}' for t in ax.get_xticklabels ()], rotation = 45)
    plt.xlabel ('Скорость ветра, м/с')
    plt.ylabel ('Повторяемость')
    plt.xlim(0, 30)
    plt.subplot(1,2,2)
    ax = df4decade['wgust'].value_counts(normalize = True).sort_index().plot.bar(ax = plt.gca(), color= 'red', alpha = 0.5, label = 'wgust')
    ax.set_xticklabels ([f'{float (t.get_text ()):g}' for t in ax.get_xticklabels ()], rotation = 45)
    plt.xlabel ('Скорость ветра, м/с')
    plt.ylabel ('Повторяемость')
    plt.suptitle(f'{decade_start} - {decade_start + 10}')
    plt.xlim(0, 30)

### Динамика повторяемости подозрительных значений во времени

Отслеживаем по годам, как меняется повторяемость подозрительных значений максимальной скорости ветра.

**Инструменты:** [DataFrame.groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html), [Series.value_counts](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) и [pandas.concat](https://pandas.pydata.org/docs/reference/api/pandas.concat.html).

In [ ]:
counts = df.groupby (df.index.year)['wgust'].value_counts(normalize = True)

years = counts.index.get_level_values(0).unique()
vals  = counts.index.get_level_values(1).unique() 

counts = pd.concat([pd.DataFrame (counts[year]).T.reindex(sorted(vals), axis=1) for year in years]).fillna(0)
counts['year'] = years
counts = counts.set_index ('year')
counts.loc[:, 13:15].plot(xlabel = 'Год', ylabel = 'Повторяемость',
                          title = 'Повторяемость wgust 13-15 м/с')



### Идентификация выбросов

Ищем выбросы с помощью диаграмм «ящик с усами» (boxplot).

**Инструменты:** [DataFrame.plot.box](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.box.html) (псевдоним [DataFrame.boxplot](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.boxplot.html)) и [seaborn.boxplot](https://seaborn.pydata.org/generated/seaborn.boxplot.html).

In [ ]:
# Ящики с усами
#df.plot.box()
df[['ta', 'td']].plot.box(ylabel = 'Скорость ветра, м/с')

In [ ]:
df.plot(kind='box', subplots=True,  figsize=(15, 5))
plt.subplots_adjust(wspace=0.9)
plt.suptitle ('Распределения признаков (ящики с усами)') 

## Двумерный анализ данных (bivariate data analysis)

Переходим к совместному анализу пар признаков и их взаимосвязей.

**Инструменты:** статистика — [pandas](https://pandas.pydata.org/docs/reference/index.html), визуализация — [Matplotlib](https://matplotlib.org/stable/api/index.html) и [Seaborn](https://seaborn.pydata.org/api.html).

### Корреляция

Считаем матрицу корреляций и отображаем её в виде тепловой карты.

**Инструменты:** [DataFrame.corr](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html) и [seaborn.heatmap](https://seaborn.pydata.org/generated/seaborn.heatmap.html).

In [ ]:
corrs = df.corr()

display (corrs)

In [ ]:


s = sns.heatmap (corrs, annot=True, annot_kws={'fontsize': 'xx-small'})
plt.title ('Матрица корреляций')
#plt.set(s,  fontsize = 'x-small')

### Диаграммы рассеяния

Строим диаграммы рассеяния для оценки связи между парами признаков.

**Инструменты:** [matplotlib.pyplot.scatter](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html); расширенный вариант с маргинальными распределениями — [seaborn.jointplot](https://seaborn.pydata.org/generated/seaborn.jointplot.html).

In [ ]:
plt.figure()
plt.scatter (df['ta'], df['td'], label = 'наблюдения')
#plt.scatter (df['wvel'], df['wgust'])
y_lim = plt.ylim()
display(y_lim)
plt.plot(y_lim, y_lim, '-k', label = 'Ta = Td')
plt.xlabel ('Ta, °C')
plt.ylabel ('Td, °C')
plt.title ('Температура воздуха и точка росы')
plt.legend()
plt.gca().set_aspect('equal')

### Шестиугольные корзины

Строим шестиугольные корзины (hexbin) для плотной визуализации большого числа точек.

**Инструменты:** [matplotlib.pyplot.hexbin](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hexbin.html), а также двумерные графики Seaborn — [seaborn.jointplot](https://seaborn.pydata.org/generated/seaborn.jointplot.html), [seaborn.boxplot](https://seaborn.pydata.org/generated/seaborn.boxplot.html) и [seaborn.pairplot](https://seaborn.pydata.org/generated/seaborn.pairplot.html).

In [ ]:
#plt.hexbin(df['wvel'], df['wgust'], gridsize=50, mincnt = 2)

plt.hexbin(df['ta'], df['td'], gridsize=50, mincnt = 1)
plt.xlabel ('Ta, °C')
plt.ylabel ('Td, °C')
plt.title ('Совместное распределение Ta и Td (hexbin)')
plt.gca().set_aspect('equal')
plt.grid()
plt.colorbar(label = 'Число наблюдений')
# plt.xlim((0, 50))
# plt.ylim((0, 50))

In [ ]:
def month_to_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'


df['season'] = [month_to_season (m) for m in df.index.month] 

sns.jointplot(df, x = 'ta', y = 'td', hue = 'season', palette="tab10") #, kind = 'hex', color = 'blue') palette="tab10") #
plt.gca().set_aspect('equal')
plt.grid()

In [ ]:
# Ящики с усами для двумерного анализа
sns.boxplot(df, x = 'tcc', y = 'rh')
plt.xlabel ('Балл облачности')
plt.ylabel ('Относительная влажность, %')
plt.title ('Влажность воздуха в зависимости от балла облачности')

In [ ]:
sns.pairplot (df[['ps', 'psl']])

### Многомерный анализ данных

Строим сетку графиков (FacetGrid) и смотрим зависимость давления на станции и на уровне моря в разных диапазонах температуры.

**Инструменты:** [pandas.cut](https://pandas.pydata.org/docs/reference/api/pandas.cut.html) для разбиения температуры на интервалы и [seaborn.FacetGrid](https://seaborn.pydata.org/generated/seaborn.FacetGrid.html) для построения сетки графиков.

In [ ]:
bins = np.arange (-30,30,10) 
labels = [f'{bins[i]} - {bins[i+1]}' for i in range (len(bins)-1)]
df['ta_range'] = pd.cut(df['ta'], bins=bins, labels=labels, right=True)


g = sns.FacetGrid(df, col = 'ta_range', height=4, aspect = 1)  

g.map(plt.scatter, 'ps', 'psl') #, gridsize=20, mincnt = 1)
g.set_axis_labels ('ps, гПа', 'psl, гПа')